# Importing the packages and data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from texttable import Texttable
import latextable

In [3]:
import sys
sys.path.insert(1, '../sar_dirichlet')
import dirichlet_regression

In [5]:
from func_test import cos_similarity, create_features_matrices, rmse_aitchison, cosine_similarity_aitchison, r2_aitchison_adjusted

# Performing the Dirichlet regression

In [35]:
Y_occitanie = pd.read_csv('Data Dirichlet/occitanie/Y_occitanie.csv', sep=';')

In [36]:
X = np.array(pd.read_csv('Data Dirichlet/occitanie/X_occitanie_final.csv'))
Y = np.array(Y_occitanie)

In [37]:
X.shape

(207, 13)

In [27]:
#X = StandardScaler().fit(X).transform(X)
#X = MinMaxScaler().fit(X).transform(X)

In [38]:
#Z = np.ones((207,1))
#gamma_0 = [0.]
Z = np.copy(X)
Z = Z[:,2:4] #we select only the columns with a significant value at the Wald test (p-value > 0.05)
gamma_0 = Z.shape[1] * [0.]

In [39]:
n,K = X.shape
J = Y.shape[1]

In [40]:
K*J

39

In [21]:
K

13

## Without spatial

In [41]:
%%time
dirichRegressor = dirichlet_regression.dirichletRegressor()
dirichRegressor.fit(X, Y, parametrization='alternative', gamma_0=gamma_0, Z=Z)

Optimization terminated successfully.
Wall time: 62.5 ms


In [42]:
print('R2:',r2_score(Y,dirichRegressor.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor.mu)))
print('AIC:',-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y)+2*30)
print('Cos similarity:',cos_similarity(Y,dirichRegressor.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor.mu))

R2: 0.42226389007226456
RMSE: 0.08466969629764724
Cross-entropy: -1.0508642758469176
AIC: -916.6733781360125
Cos similarity: 0.9722244301492748
RMSE_A: 0.40289224833295295


In [43]:
len((dirichRegressor.beta).flatten())+len(gamma_0)

30

In [44]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor.mu,30))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor.mu))

Aitchison metrics
R2: 0.29114764301342766
Cos similarity: 0.7289456608831856


## With spatial

### Distance

In [45]:
W = np.copy(distance_matrix)
W[W > 35000] = 0

In [46]:
# inverse distance
W[W>0] = 1/W[W>0]

# row-normalize
W = W/W.sum(axis=1)[:,None]

In [47]:
np.mean([np.sum(W[i]!=0) for i in range(207)])

12.096618357487923

In [48]:
gamma_0 = [-2.95, 4.8]

In [49]:
beta_0 = np.array([[-2.49710710e+00, -1.68536902e+00],
       [ 5.55500384e-02, -3.22378326e-01],
       [ 2.97479297e-01,  7.14892149e-01],
       [-9.92275074e-01, -4.89698477e-01],
       [ 4.69438983e-01, -1.52666860e-01],
       [-3.38862522e-01, -3.40651195e-01],
       [ 4.15632539e-01,  2.65599053e-01],
       [-2.00136888e-01, -7.30792593e-02],
       [ 1.23402296e-01,  5.84038038e-02],
       [ 4.47972661e+00,  4.47628556e+00], #unemp_rate
       [-6.31824798e-03, -2.49309772e-03], #employ_evol
       [-5.61073970e-01, -8.60807378e-02], #owner_rate
       [ 3.30576021e+00,  1.80937582e+00], #income_rate
       [-9.30703518e-02, -5.75768737e-02]]) #foreign

In [50]:
%%time
dirichRegressor_s1 = dirichlet_regression.dirichletRegressor(spatial=True, maxiter=5000, maxfun=500000)
dirichRegressor_s1.fit(X, Y, W=W, parametrization='alternative', gamma_0=gamma_0, beta_0 = beta_0, rho_0 = 0.99, Z=Z)

CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Wall time: 52.7 s


In [449]:
print('R2:',r2_score(Y,dirichRegressor_s1.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor_s1.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor_s1.mu)))
print('AIC:',-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor_s1.mu,dirichRegressor_s1.phi,Y)+2*53)
print('Cos similarity:',cos_similarity(Y,dirichRegressor_s1.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor_s1.mu))

R2: 0.49277922070066266
RMSE: 0.07925035674529514
Cross-entropy: -1.0469351979318682
AIC: -888.3248885197031
Cos similarity: 0.9755725774994097
RMSE_A: 0.3973880529535168


In [53]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor_s1.mu,31))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor_s1.mu))

Aitchison metrics
R2: 0.3711686912547363
Cos similarity: 0.7514835880260777


## Cross-entropy

### Not spatial

In [54]:
%%time
dirichRegressor_ce = dirichlet_regression.dirichletRegressor(spatial=False, maxiter=5000, maxfun=10000)
dirichRegressor_ce.fit(X, Y, parametrization='alternative', gamma_0=gamma_0, Z=Z, loss='crossentropy')

Optimization terminated successfully.
Wall time: 53.4 ms


In [55]:
# without 'pop'
print('R2:',r2_score(Y,dirichRegressor_ce.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor_ce.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor_ce.mu)))
print('Cos similarity:',cos_similarity(Y,dirichRegressor_ce.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor_ce.mu))

R2: 0.4476515324399187
RMSE: 0.08277594662100392
Cross-entropy: -1.0497159964287022
Cos similarity: 0.9731408549667323
RMSE_A: 0.401828735470348


In [56]:
k = len(dirichRegressor_ce.beta.flatten())
print('R2 adjusted : ', 1 - (1 - r2_score(Y,dirichRegressor_ce.mu)) * (n-1)/(n-k))

R2 adjusted :  0.36433640046158233


In [57]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor_ce.mu,k))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor_ce.mu))

Aitchison metrics
R2: 0.30701960484982416
Cos similarity: 0.7365956927899285


### Spatial

In [58]:
%%time
dirichRegressor_s1_ce = dirichlet_regression.dirichletRegressor(spatial=True, maxiter=5000, maxfun=500000)
dirichRegressor_s1_ce.fit(X, Y, W=W, parametrization='alternative', gamma_0=gamma_0, Z=Z, loss='crossentropy')

Optimization terminated successfully.
Wall time: 1.27 s


In [59]:
# without 'pop'
print('R2:',r2_score(Y,dirichRegressor_s1_ce.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor_s1_ce.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor_s1_ce.mu)))
print('Cos similarity:',cos_similarity(Y,dirichRegressor_s1_ce.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor_s1_ce.mu))

R2: 0.5181418843034878
RMSE: 0.07729177325912936
Cross-entropy: -1.0456541662162306
Cos similarity: 0.9766712440972178
RMSE_A: 0.39336855095913154


In [47]:
k = len(dirichRegressor_s1_ce.beta.flatten()) + 1
print('R2 adjusted : ', 1 - (1 - r2_score(Y,dirichRegressor_s1_ce.mu)) * (n-1)/(n-k))

R2 adjusted :  0.4446057628186578


In [60]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor_s1_ce.mu,k))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor_s1_ce.mu))

Aitchison metrics
R2: 0.3806886923072291
Cos similarity: 0.7477961863805495


# With spatial contiguity

In [61]:
W_occitanie = pd.read_csv('Data Dirichlet/occitanie/W_elections_5nn.csv', sep=' ', header=None)
W = np.array(W_occitanie)

## Dirichlet

In [62]:
%%time
dirichRegressor_s2 = dirichlet_regression.dirichletRegressor(spatial=True, maxiter=5000, maxfun=500000)
dirichRegressor_s2.fit(X, Y, W=W, parametrization='alternative', gamma_0=gamma_0, Z=Z)

CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Wall time: 2min 33s


In [57]:
len(dirichRegressor_s2.beta.flatten()) + len(dirichRegressor_s2.gamma.flatten()) + 1

31

In [58]:
# without 'pop'
print('R2:',r2_score(Y,dirichRegressor_s2.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor_s2.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor_s2.mu)))
print('AIC:',-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor_s2.mu,dirichRegressor_s2.phi,Y)+2*31)
print('Cos similarity:',cos_similarity(Y,dirichRegressor_s2.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor_s2.mu))

R2: 0.47341586963239113
RMSE: 0.08080267367650393
Cross-entropy: -1.0486254593112267
AIC: -933.5554456429596
Cos similarity: 0.9742871356316848
RMSE_A: 0.39469394415329023


In [63]:
k = len(dirichRegressor_s2.beta.flatten()) + len(dirichRegressor_s2.gamma.flatten()) + 1
print('R2 adjusted : ', 1 - (1 - r2_score(Y,dirichRegressor_s2.mu)) * (n-1)/(n-k))

R2 adjusted :  0.3825126395465719


In [65]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor_s2.mu,k))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor_s2.mu))

Aitchison metrics
R2: 0.32462311353881546
Cos similarity: 0.7315353648346418


## Cross-entropy

In [66]:
%%time
dirichRegressor_s2_ce = dirichlet_regression.dirichletRegressor(spatial=True, maxiter=5000, maxfun=500000)
dirichRegressor_s2_ce.fit(X, Y, W=W, parametrization='alternative', gamma_0=gamma_0, Z=Z, loss='crossentropy')

Optimization terminated successfully.
Wall time: 1.09 s


In [51]:
# without 'pop'
print('R2:',r2_score(Y,dirichRegressor_s2_ce.mu))
print('RMSE:',mean_squared_error(Y,dirichRegressor_s2_ce.mu,squared=False))
print('Cross-entropy:',1/n*np.sum(Y*np.log(dirichRegressor_s2_ce.mu)))
print('Cos similarity:',cos_similarity(Y,dirichRegressor_s2_ce.mu))
print('RMSE_A:', rmse_aitchison(Y,dirichRegressor_s2_ce.mu))

R2: 0.49316813218417166
RMSE: 0.07924751107449089
Cross-entropy: -1.0477713693884736
Cos similarity: 0.975021022020977
RMSE_A: 0.39707684733840404


In [52]:
k = len(dirichRegressor_s2_ce.beta.flatten()) + 1
print('R2 adjusted : ', 1 - (1 - r2_score(Y,dirichRegressor_s2_ce.mu)) * (n-1)/(n-k))

R2 adjusted :  0.41344177095471546


In [67]:
print('Aitchison metrics')
print('R2:', r2_aitchison_adjusted(Y,dirichRegressor_s2_ce.mu,k))
print('Cos similarity:', cosine_similarity_aitchison(Y,dirichRegressor_s2_ce.mu))

Aitchison metrics
R2: 0.3276319552271588
Cos similarity: 0.7173992137051547
